# Training energy: K40 Sign-CatBoost, K40 MLP, and K40-Adapt LoRA

This notebook **refits the actual models** with the paper's split seed 67 and training seeds 42, 123, 2026. It measures one three-seed batch for each method: Sign-CatBoost, MLP width 256, and LoRA ranks 1, 2, 4, 8. No local-test predictions or labels are used. CatBoost fits the combined 90% training + validation rows; MLP and LoRA fit the 70% training rows and select using the 20% validation rows.

`powermetrics` needs administrator access, so start its sampler in **your own Terminal** when the cell below prints the command. The notebook never asks for or stores your password. The measurement cell waits 60 seconds for an idle reference, then trains all six method/rank batches. Keep the laptop plugged in and avoid other heavy workloads. The power readings are macOS estimates for CPU + GPU + ANE, not wall-plug electricity. The notebook does not install anything or change the manuscript.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import contextlib, io, json, platform, re, shlex, sys, time

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "splits/manifest.json").exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from common import _digest
from catboost.pipeline import train as catboost_train
from mlp.train import Config as MLPConfig, fit as mlp_fit
from lora.kryptonet import load_kryptonet
from lora.model import apply_lora, count_trainable
from lora.train import train as lora_train, logits as lora_logits

SEEDS = (42, 123, 2026)
RANKS = (1, 2, 4, 8)
assert platform.system() == "Darwin", "This measurement uses macOS powermetrics."
assert Path("/usr/bin/powermetrics").exists(), "macOS powermetrics is unavailable."
assert (ROOT / "kryptonet40.pt").exists(), "Place the course-supplied checkpoint at the repository root."
RUN = ROOT / "results/notebook_reruns" / ("energy-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
RUN.mkdir(parents=True, exist_ok=False)
POWER_LOG = RUN / "powermetrics.txt"
MARKERS = RUN / "timing_markers.json"
torch.set_num_threads(1)
print("Hardware:", platform.machine(), "| PyTorch threads:", torch.get_num_threads())
print("Split seed: 67 | Training seeds:", SEEDS, "| LoRA ranks:", RANKS)
print("New output directory:", RUN)
print("No training or power sampling has started yet.")

Hardware: arm64 | PyTorch threads: 1
Split seed: 67 | Training seeds: (42, 123, 2026) | LoRA ranks: (1, 2, 4, 8)
New output directory: /Users/hardenzhao/Kryptonite_v3.0/results/notebook_reruns/energy-20261119T141558436875Z
No training or power sampling has started yet.


## Verify and load training/validation rows

The helper below checks the shared manifest, then selects only the `train` and `validation` indices. It does not index or score the 10% local-test partition. It prepares the arrays before power sampling, so array loading does not inflate the training measurements.

In [2]:
def load_train_val(name):
    manifest = json.loads((ROOT / "splits/manifest.json").read_text())
    assert manifest["seed"] == 67
    entry = manifest["datasets"][name]
    x_path = ROOT / "Datasets" / f"kryptonite-{name}-X.npy"
    y_path = ROOT / "Datasets" / f"kryptonite-{name}-y.npy"
    split_path = ROOT / "splits" / f"{name}.npz"
    for path, key in ((x_path, "x_sha256"), (y_path, "y_sha256"), (split_path, "split_sha256")):
        assert _digest(path) == entry[key], f"Manifest mismatch: {path}"
    with np.load(split_path, allow_pickle=False) as split:
        train_idx = split["train"].copy()
        val_idx = split["validation"].copy()
        test_idx = split["test"].copy()  # indices only; no test features or labels are indexed
    assert not np.intersect1d(train_idx, val_idx).size
    assert not np.intersect1d(train_idx, test_idx).size
    assert not np.intersect1d(val_idx, test_idx).size
    x = np.load(x_path, allow_pickle=False, mmap_mode="r")
    y = np.load(y_path, allow_pickle=False, mmap_mode="r")
    return x[train_idx], y[train_idx], x[val_idx], y[val_idx]

mlp_x, mlp_y, mlp_vx, mlp_vy = load_train_val("40")
adapt_x, adapt_y, adapt_vx, adapt_vy = load_train_val("40-adapt")
tx = torch.from_numpy(np.ascontiguousarray(adapt_x, dtype=np.float32))
ty = torch.from_numpy(np.ascontiguousarray(adapt_y, dtype=np.float32))
vx = torch.from_numpy(np.ascontiguousarray(adapt_vx, dtype=np.float32))
vy = torch.from_numpy(np.ascontiguousarray(adapt_vy, dtype=np.float32))
print("K40 MLP: train", len(mlp_y), "/ validation", len(mlp_vy))
print("K40-Adapt LoRA: train", len(ty), "/ validation", len(vy))
print("Sign-CatBoost uses 28,800 combined K40 training + validation rows via its existing pipeline.")

K40 MLP: train 22400 / validation 6400
K40-Adapt LoRA: train 2800 / validation 800
Sign-CatBoost uses 28,800 combined K40 training + validation rows via its existing pipeline.


## Start the power sampler in Terminal

Run the printed command in a **separate Terminal window**. Enter the macOS administrator password there if prompted. Wait for 2–3 samples to appear, then run the next notebook cell. Leave the sampler running throughout the 60-second idle period and all model fits. After the training cell finishes, stop the Terminal sampler with **Ctrl+C**. Shell redirection creates the log as your user, so the notebook can read it. The sampler uses the built-in `/usr/bin/powermetrics` at 1-second intervals; no package installation is involved.

In [5]:
print("Run this in Terminal, then execute the next notebook cell:")
print("sudo /usr/bin/powermetrics -s cpu_power -i 1000 -b 1 > " + shlex.quote(str(POWER_LOG)))
print("When training is finished, press Ctrl+C in that Terminal before analyzing the log.")

Run this in Terminal, then execute the next notebook cell:
sudo /usr/bin/powermetrics -s cpu_power -i 1000 -b 1 > /Users/hardenzhao/Kryptonite_v3.0/results/notebook_reruns/energy-20261119T141558436875Z/powermetrics.txt
When training is finished, press Ctrl+C in that Terminal before analyzing the log.


## Measure idle power and refit all six batches

The notebook records UTC boundaries for the 60-second idle window and each three-seed batch. Each method uses the existing model and training implementation. CatBoost and MLP write normal training artifacts into this new, Git-ignored run directory. LoRA saves the selected weights for each seed. The cell does **not** evaluate the local test. It may take several minutes. If it fails partway through, the completed timing records remain in `timing_markers.json`.

In [7]:
assert POWER_LOG.exists(), "Start the Terminal sampler first; the expected power log does not exist."
def sample_count(path):
    return len(re.findall(r"^\*\*\* Sampled system activity \(", path.read_text(errors="replace"), flags=re.M))

deadline = time.monotonic() + 20
while sample_count(POWER_LOG) < 2 and time.monotonic() < deadline:
    time.sleep(1)
assert sample_count(POWER_LOG) >= 2, "No powermetrics samples appeared within 20 seconds. Check the Terminal sampler."

records = {"split_seed": 67, "training_seeds": list(SEEDS), "ranks": list(RANKS),
           "power_field": "Combined Power (CPU + GPU + ANE)", "sample_interval_ms": 1000,
           "idle": {}, "groups": [], "jobs": [], "run_dir": str(RUN)}
def stamp():
    return datetime.now(timezone.utc).isoformat()
def persist():
    MARKERS.write_text(json.dumps(records, indent=2) + "\n")
def batch(name, job):
    print("Starting", name, flush=True)
    time.sleep(3)  # separate adjacent groups in the 1-second power log
    group = {"name": name, "started_utc": stamp(), "finished_utc": None, "status": "running"}
    records["groups"].append(group)
    persist()
    try:
        for seed in SEEDS:
            started = time.perf_counter()
            result = job(seed)
            duration = time.perf_counter() - started
            records["jobs"].append({"group": name, "seed": seed, "duration_s": duration, **result})
            persist()
            print(f"  seed {seed}: {duration:.2f} s", flush=True)
        group["status"] = "completed"
    except BaseException:
        group["status"] = "interrupted"
        raise
    finally:
        group["finished_utc"] = stamp()
        persist()

records["idle"] = {"started_utc": stamp(), "finished_utc": None}
persist()
print("Measuring 60 seconds of idle power; leave the machine alone if possible.")
time.sleep(60)
records["idle"]["finished_utc"] = stamp()
persist()

def catboost_job(seed):
    manifest = catboost_train("40", "training+validation", seed,
                              RUN / "catboost" / f"seed{seed}")
    return {"training_rows": manifest["training_count"],
            "fit_seconds": manifest["training_seconds"]}

def mlp_job(seed):
    cfg = MLPConfig(dataset="40", hidden_sizes=(256, 256), seed=seed,
                    purpose="width_comparison")
    with contextlib.redirect_stdout(io.StringIO()):
        summary = mlp_fit(cfg, mlp_x, mlp_y, mlp_vx, mlp_vy,
                          RUN / "mlp" / f"seed{seed}")
    return {"training_rows": summary["training_size"],
            "validation_rows": summary["validation_size"],
            "best_epoch": summary["best_epoch"], "epochs_run": summary["epochs_run"],
            "validation_accuracy": summary["validation"]["accuracy"]}

def lora_job(rank, seed):
    torch.manual_seed(seed)
    base = load_kryptonet(ROOT / "kryptonet40.pt")
    model = apply_lora(base, rank=rank, alpha=8.0, train_head=True)
    model, best_epoch, history = lora_train(
        model, tx, ty, vx, vy, lr=1e-3, max_epochs=300,
        patience=40, batch_size=128, seed=seed)
    out = RUN / "lora" / f"rank{rank}_seed{seed}.pt"
    out.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out)
    val_accuracy = float(((lora_logits(model, vx) > 0).float() == vy).float().mean())
    return {"training_rows": len(ty), "validation_rows": len(vy),
            "rank": rank, "alpha": 8.0, "trainable_params": count_trainable(model),
            "best_epoch": best_epoch, "epochs_run": len(history),
            "validation_accuracy": val_accuracy}

batch("Sign-CatBoost K40", catboost_job)
batch("MLP width 256 K40", mlp_job)
for rank in RANKS:
    batch(f"LoRA rank {rank} K40-Adapt", lambda seed, rank=rank: lora_job(rank, seed))

print("All 18 fits completed. Stop the Terminal sampler with Ctrl+C, then run the analysis cell.")
print("Timing markers:", MARKERS)

Measuring 60 seconds of idle power; leave the machine alone if possible.
Starting Sign-CatBoost K40
  seed 42: 26.03 s
  seed 123: 26.57 s
  seed 2026: 26.44 s
Starting MLP width 256 K40
  seed 42: 5.06 s
  seed 123: 4.41 s
  seed 2026: 3.97 s
Starting LoRA rank 1 K40-Adapt
  seed 42: 1.76 s
  seed 123: 1.94 s
  seed 2026: 2.09 s
Starting LoRA rank 2 K40-Adapt
  seed 42: 1.58 s
  seed 123: 1.81 s
  seed 2026: 2.24 s
Starting LoRA rank 4 K40-Adapt
  seed 42: 1.67 s
  seed 123: 1.64 s
  seed 2026: 1.73 s
Starting LoRA rank 8 K40-Adapt
  seed 42: 1.65 s
  seed 123: 1.65 s
  seed 2026: 1.73 s
All 18 fits completed. Stop the Terminal sampler with Ctrl+C, then run the analysis cell.
Timing markers: /Users/hardenzhao/Kryptonite_v3.0/results/notebook_reruns/energy-20261119T141558436875Z/timing_markers.json


## Parse samples and calculate energy

The text parser uses the sample timestamp, actual sample duration, and the `Combined Power (CPU + GPU + ANE)` field. A sample must lie fully inside a marked period with a 1-second boundary guard, because the text timestamp is printed to the nearest second. Power is averaged using sample duration; batch energy is the interior mean power multiplied by the exact marked batch time. This is an **estimate** for the full batch, with 1-second sampling uncertainty. The idle-subtracted value estimates incremental SoC energy above this session's idle baseline; it is not total laptop electricity. Batches with fewer than five interior samples or under 50% interior coverage are flagged as low quality.

In [8]:
from datetime import timedelta
assert MARKERS.exists(), "Run the training cell first."
records = json.loads(MARKERS.read_text())
assert all(g["status"] == "completed" for g in records["groups"]), "One or more training batches did not finish."
raw = POWER_LOG.read_text(errors="replace")
header = re.compile(r"^\*\*\* Sampled system activity \((.*?)\) \(([0-9.]+)ms elapsed\) \*\*\*$", re.M)
power_patterns = {key: re.compile(r"^" + re.escape(label) + r": ([0-9.]+) mW$", re.M)
                  for key, label in (("cpu_mw", "CPU Power"), ("gpu_mw", "GPU Power"),
                                     ("ane_mw", "ANE Power"),
                                     ("combined_mw", "Combined Power (CPU + GPU + ANE)"))}
matches = list(header.finditer(raw))
samples = []
for i, match in enumerate(matches):
    block = raw[match.end():matches[i + 1].start() if i + 1 < len(matches) else len(raw)]
    values = {key: pattern.search(block) for key, pattern in power_patterns.items()}
    if any(value is None for value in values.values()):
        continue  # allow an incomplete last sample if the sampler was just stopped
    end = datetime.strptime(match.group(1), "%a %b %d %H:%M:%S %Y %z")
    duration = float(match.group(2)) / 1000
    samples.append({"end_utc": end, "start_utc": end - timedelta(seconds=duration),
                    "duration_s": duration,
                    **{key: float(value.group(1)) for key, value in values.items()}})
assert samples, "No complete CPU/GPU/ANE power samples were found. Check the powermetrics log."

def measure(period, guard_s=1.0):
    start = datetime.fromisoformat(period["started_utc"])
    end = datetime.fromisoformat(period["finished_utc"])
    elapsed_s = (end - start).total_seconds()
    inside = [s for s in samples if s["start_utc"] >= start + timedelta(seconds=guard_s)
              and s["end_utc"] <= end - timedelta(seconds=guard_s)]
    observed_s = sum(s["duration_s"] for s in inside)
    if not inside:
        return {"duration_s": elapsed_s, "sample_count": 0,
                "sampled_s": 0.0, "coverage": 0.0,
                "combined_W": float("nan"), "cpu_W": float("nan")}
    average = lambda field: sum(s[field] * s["duration_s"] for s in inside) / observed_s / 1000
    return {"duration_s": elapsed_s, "sample_count": len(inside),
            "sampled_s": observed_s, "coverage": observed_s / elapsed_s,
            "combined_W": average("combined_mw"), "cpu_W": average("cpu_mw")}

idle = measure(records["idle"])
assert idle["sample_count"] >= 30, "The 60-second idle reference has too few fully contained samples."
rows = []
for group in records["groups"]:
    result = measure(group)
    result["group"] = group["name"]
    result["gross_Wh_est"] = result["combined_W"] * result["duration_s"] / 3600
    result["above_idle_Wh_est"] = (result["combined_W"] - idle["combined_W"]) * result["duration_s"] / 3600
    result["quality"] = "review short batch" if result["sample_count"] < 5 or result["coverage"] < 0.5 else "OK"
    rows.append(result)
summary = pd.DataFrame(rows).set_index("group")
jobs = pd.DataFrame(records["jobs"])
summary.to_csv(RUN / "energy_summary.csv")
jobs.to_csv(RUN / "training_runs.csv", index=False)
pd.DataFrame([{**s, "start_utc": s["start_utc"].isoformat(),
               "end_utc": s["end_utc"].isoformat()} for s in samples]).to_csv(RUN / "power_samples.csv", index=False)
print(f"Idle: {idle['combined_W']:.3f} W combined, {idle['cpu_W']:.3f} W CPU; "
      f"{idle['sample_count']} interior samples from {idle['duration_s']:.1f} s.")
display(summary[["duration_s", "sample_count", "coverage", "combined_W",
                 "gross_Wh_est", "above_idle_Wh_est", "quality"]].round(4))
display(jobs[["group", "seed", "duration_s", "epochs_run"] if "epochs_run" in jobs else
             ["group", "seed", "duration_s"]].round(3))
print("Saved raw samples, timing markers, per-seed durations, and energy estimates in", RUN)

Idle: 2.494 W combined, 1.800 W CPU; 56 interior samples from 60.0 s.


,duration_s,sample_count,coverage,combined_W,gross_Wh_est,above_idle_Wh_est,quality
group,,,,,,,
Sign-CatBoost K40,79.0379,75,0.9607,5.4972,0.1207,0.0659,OK
MLP width 256 K40,13.4424,11,0.8281,7.0708,0.0264,0.0171,OK
LoRA rank 1 K40-Adapt,5.7909,2,0.3498,7.8065,0.0126,0.0085,review short batch
LoRA rank 2 K40-Adapt,5.6293,3,0.5388,7.7744,0.0122,0.0083,review short batch
LoRA rank 4 K40-Adapt,5.0533,2,0.4005,7.6965,0.0108,0.0073,review short batch
LoRA rank 8 K40-Adapt,5.0359,2,0.4016,7.6835,0.0107,0.0073,review short batch


,group,seed,duration_s,epochs_run
0,Sign-CatBoost K40,42,26.025,NaN
1,Sign-CatBoost K40,123,26.570,NaN
2,Sign-CatBoost K40,2026,26.441,NaN
3,MLP width 256 K40,42,5.061,38.0
4,MLP width 256 K40,123,4.405,40.0
5,MLP width 256 K40,2026,3.974,36.0
6,LoRA rank 1 K40-Adapt,42,1.755,58.0
7,LoRA rank 1 K40-Adapt,123,1.941,62.0
8,LoRA rank 1 K40-Adapt,2026,2.092,66.0
9,LoRA rank 2 K40-Adapt,42,1.578,53.0


Saved raw samples, timing markers, per-seed durations, and energy estimates in /Users/hardenzhao/Kryptonite_v3.0/results/notebook_reruns/energy-20261119T141558436875Z
